# Convolutional Neural Networks (CNNs)

This notebook covers:
- Why CNNs are designed for images
- Convolution operation: filters, feature maps, padding, stride
- Pooling layers for dimensionality reduction
- Building CNN architectures with Conv2D and MaxPooling2D
- Training CNNs on Fashion-MNIST

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")

## Why Convolutions Matter

Dense layers treat each pixel independently. Convolutions exploit **spatial structure**:

| Aspect | Dense Layer | Convolutional Layer |
|--------|-------------|-------------------|
| Connectivity | Every input → every neuron | Local receptive field (3×3, 5×5) |
| Parameters | Thousands per neuron | Shared across all positions |
| Translation Invariance | No | Yes - detects patterns anywhere |
| Use Case | Tabular data | Images, spatial structure |

**Key Benefits:**
- **Parameter sharing**: Same filter reused at every position
- **Local connectivity**: Each output depends on small input region
- **Translation invariance**: Same pattern detected anywhere
- **Efficient**: Far fewer parameters than dense networks on images

In [ ]:
# Load Fashion-MNIST dataset
print("Loading Fashion-MNIST...")
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

class_names = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

print(f"Training shape: {X_train.shape}, Test shape: {X_test.shape}")

# Normalize and add channel dimension for Conv2D
X_train = X_train[..., np.newaxis] / 255.0
X_test = X_test[..., np.newaxis] / 255.0

print(f"After reshape: Train {X_train.shape}, Test {X_test.shape}")

# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i, :, :, 0], cmap='gray')
    ax.set_title(class_names[y_train[i]])
    ax.axis('off')
plt.tight_layout()
plt.show()

## Building a CNN Model

Architecture pattern:
1. **Conv2D**: Extract features from images (multiple filters learn different patterns)
2. **MaxPooling2D**: Downsample (reduce spatial dimensions, keep dominant features)
3. Repeat 2-3 times to build hierarchical features
4. **Flatten**: Convert to 1D vector
5. **Dense**: Classification layers

In [ ]:
# Build CNN model
model_cnn = tf.keras.Sequential([
    # Block 1: Conv + Pooling
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    # Block 2: Conv + Pooling
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    # Flatten and Dense layers
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Compile
model_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("CNN Model Architecture:")
model_cnn.summary()

In [ ]:
# Train CNN
print("Training CNN on Fashion-MNIST...")
history_cnn = model_cnn.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    verbose=0
)

# Evaluate
test_loss, test_acc = model_cnn.evaluate(X_test, y_test, verbose=0)
print(f"\nCNN Test Accuracy: {test_acc:.4f}")
print(f"CNN Test Loss: {test_loss:.4f}")

# Compare with dense network
model_dense = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model_dense.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_dense = model_dense.fit(X_train, y_train, epochs=10, batch_size=128, 
                                 validation_split=0.2, verbose=0)
test_loss_dense, test_acc_dense = model_dense.evaluate(X_test, y_test, verbose=0)

print(f"Dense Network Test Accuracy: {test_acc_dense:.4f}")
print(f"\nCNN vs Dense: CNN is {(test_acc - test_acc_dense)*100:.2f}% {'better' if test_acc > test_acc_dense else 'worse'}")

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Training history
axes[0].plot(history_cnn.history['accuracy'], label='CNN Train', marker='o', markersize=3)
axes[0].plot(history_cnn.history['val_accuracy'], label='CNN Val', marker='s', markersize=3)
axes[0].plot(history_dense.history['accuracy'], label='Dense Train', marker='^', markersize=3)
axes[0].plot(history_dense.history['val_accuracy'], label='Dense Val', marker='d', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('CNN vs Dense Network Training')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test accuracy comparison
models = ['CNN', 'Dense']
accuracies = [test_acc, test_acc_dense]
axes[1].bar(models, accuracies, alpha=0.7, color=['#1f77b4', '#ff7f0e'])
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Test Accuracy Comparison')
axes[1].set_ylim([0.88, 0.95])
for i, v in enumerate(accuracies):
    axes[1].text(i, v + 0.001, f'{v:.4f}', ha='center', fontweight='bold')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()